# Optimize LP01 nonlinear distortion by changing one input

Follow the original power-optimization notebook, replacing squeezing with
**LP01 nonlinear distortion**. Vary only peak power, starting at **100 kW**.
Keep the **200 fs Gaussian**, **LP01 launch**, **three propagated modes**,
**2.86 m step-index fiber**, and **1379.2 nm carrier**.

At each power, propagate the **same input** with nonlinearity disabled
(linear reference, n2 = 0) and enabled (nonlinear output). Compare mode 1:

\[
F_{\mathrm{NL},01}=
\frac{\left|\int A_{\mathrm{lin},1}^{*}(L,t)A_{\mathrm{NL},1}(L,t)\,dt\right|^2}
{\left(\int |A_{\mathrm{lin},1}(L,t)|^2dt\right)
 \left(\int |A_{\mathrm{NL},1}(L,t)|^2dt\right)},
\qquad D_{\mathrm{NL},01}=1-F_{\mathrm{NL},01}.
\]

**Lower DNL is better.** Zero means the LP01 fields match up to one complex
scale factor. All three modes propagate and may couple, but only LP01 is
launched and scored. The objective uses the **full LP01 pulse, without a
spectral filter**. This is classical field overlap, not photon-number squeezing.

Normalized overlap does not penalize amplitude loss. Keep the original 50%
brightness constraint, applied here to **unfiltered LP01 photons**; also report
the other modes' energy. Changing power changes pulse energy and nonlinear
interaction strength. This is not a fixed-energy preservation experiment.

Run in order with **GNLSE (Julia)**.

In [ ]:
using Pkg
workspace = let dir = abspath(@__DIR__)
    while !isfile(joinpath(dir, "notebook_helpers", "run_library.py"))
        dirname(dir) == dir && error("Open this notebook inside the Nonlinear Optics workspace.")
        dir = dirname(dir)
    end
    dir
end
ROOT = get(ENV, "DNL_PROJECT_ROOT", joinpath(workspace, "FastDifferentiableMMNLO"))
Pkg.activate(ROOT)
using PulsePropagation, FFTW, LinearAlgebra, Printf, TOML, Dates, SHA, NPZ
using PyPlot

BLAS.set_num_threads(1)
FFTW.set_num_threads(1)
run_dir = haskey(ENV, "DNL_RUN_DIR") ? ENV["DNL_RUN_DIR"] :
    mktempdir(mkpath(joinpath(workspace, "output", "organized", "runs", "Total launch power", "Gradient descent")); prefix=Dates.format(now(), "yyyy-mm-dd") * "_step_index_dnl_power_optimization_", cleanup=false)
mkpath(run_dir)
println("Results: ", run_dir)

# Load the launch, objective, descent, and plotting functions used below.
include(joinpath(workspace, "notebook_helpers", "julia", "Total launch power", "shared.jl"))
include(joinpath(workspace, "notebook_helpers", "julia", "Total launch power", "step_index_dnl_power_optimization.jl"))

## 1. Choose the control and the compute budget

Use \(u=P/(100\,\mathrm{kW})\), 50–150 kW bounds, and at least 50%
of the baseline LP01 photon count. Search with 10 mm steps and validate at 1 mm.

The learning rate and tolerance now use dimensionless DNL rather than dB.
The finite-difference spacing is still ±200 W. Each new evaluation runs
**two forward solves**, with no squeezing adjoint. Rerun from the top after
changing settings so the parameters, baseline and caches are rebuilt.
Allow up to 20 optimization iterations and 80 new objective evaluations.

In [ ]:
reference_power_W = 100_000.0
power_bounds_W = (50_000.0, 150_000.0)
minimum_brightness_fraction = 0.50
fd_step = 0.002                  # ±200 W when differentiating with respect to u
learning_rate = 0.10
max_iterations = 20
max_optimizer_evaluations = 80
gradient_tolerance = 1e-4        # DNL per unit u
max_power_change_W = 10_000.0
search_dz = 0.01                # m
validation_dz = 0.001           # m

dnl_step_tolerance = 2e-4  # absolute DNL difference, not dB
launch_coefficients = [1.0, 0.0, 0.0]

settings = Dict(
    "control" => "input Gaussian peak power (W)",
    "objective" => "minimize LP01 DNL = 1 - complex field fidelity",
    "reference_power_W" => reference_power_W,
    "power_bounds_W" => collect(power_bounds_W),
    "minimum_brightness_fraction" => minimum_brightness_fraction,
    "finite_difference_step_u" => fd_step,
    "learning_rate" => learning_rate,
    "max_iterations" => max_iterations,
    "max_optimizer_evaluations" => max_optimizer_evaluations,
    "gradient_tolerance" => gradient_tolerance,
    "max_power_change_W" => max_power_change_W,
    "search_dz_m" => search_dz,
    "validation_dz_m" => validation_dz,
    "Nt" => 2048, "time_window_ps" => 10.0, "pulse_fwhm_ps" => 0.2,
    "fiber_length_m" => 2.86, "carrier_m" => 1379.2e-9,
    "core_radius_m" => 52.5e-6, "NA" => 0.1, "num_modes" => 3,
    "objective_mode" => 1, "objective_filter" => "none",
    "linear_reference" => "same candidate input and fiber; n2 = 0",
    "brightness_definition" => "unfiltered LP01 photons relative to baseline",
    "dnl_step_tolerance" => dnl_step_tolerance,
    "launch_coefficients" => launch_coefficients,
    "cp_target_error" => 1e-3, "cp_seed" => Int(0x4d4d474e),
    "julia_version" => string(VERSION),
    "manifest_sha256" => bytes2hex(sha256(read(joinpath(ROOT, "Manifest.toml")))),
)
open(joinpath(run_dir, "settings.toml"), "w") do io
    TOML.print(io, settings)
end

## 2. Build the fixed fiber and linear reference once

Use the original three-mode fiber and CP compression. The linear and nonlinear
parameter sets differ only in n2. Use the same candidate input, grid, dispersion,
length, and propagation step in both branches.

Brightness now counts LP01 photons across the full spectrum. All-mode photon
count and modal energy fractions are additional diagnostics.

In [ ]:
time_grid = TimeGrid(settings["Nt"], settings["time_window_ps"])
domain = MMGNLSEDomain(degrees_of_freedom(:time, :space), time_grid)
properties = compute_fiber_properties(
    StepIndex(settings["carrier_m"], settings["core_radius_m"], settings["NA"]),
    Silica(), settings["num_modes"];
    polarization=:scalar, mode_basis=:LP, beta_order=5)
S_cp = cp_compress(properties.S; error=settings["cp_target_error"], seed=settings["cp_seed"])
parameters_cp = MMGNLSEParameters(domain;
    length=settings["fiber_length_m"], beta=properties.beta, S=S_cp,
    n2=properties.n2, omega0=properties.omega0, raman=properties.raman)


parameters_linear = MMGNLSEParameters(domain;
    length=settings["fiber_length_m"], beta=properties.beta, S=S_cp,
    n2=0.0, omega0=properties.omega0, raman=properties.raman)
lp01_photons = SpectralPhotonNumber(filter=ones(settings["Nt"]), modes=[1])
all_photons = SpectralPhotonNumber(filter=ones(settings["Nt"]), modes=:all)
@printf("Three propagated modes; LP01 launch and score. CP error: %.3g\n", S_cp.relative_error)
println("Mode labels: ", properties.modes.labels)

## 3. Turn the DNL calculation into a function

The common time-spacing factors cancel in the normalized overlap. Keep the
**complex fields**, including phase, and compare only mode 1. Reject zero fields.
F_input separately compares the nonlinear LP01 output with the original input.

As in the original, cache evaluations and save a CSV. Also cache endpoint fields
for the final plots. The small algebra checks below do not run fiber simulations.

In [ ]:
evaluation_cache = Dict{Tuple{Float64,Float64},NamedTuple}()
field_cache = Dict{Tuple{Float64,Float64},NamedTuple}()
evaluations = NamedTuple[]

let a = zeros(ComplexF64, 4, 3, 1)
    a[1, 1, 1] = 1
    @assert isapprox(lp01_fidelity(a, a), 1.0; atol=1e-12)
    @assert isapprox(lp01_fidelity(a, 2exp(0.7im).*a), 1.0; atol=1e-12)
    b = copy(a)
    b[:, 2, 1] .= 3  # Changes in other modes do not enter this objective.
    @assert isapprox(lp01_fidelity(a, b), 1.0; atol=1e-12)
    b = zeros(ComplexF64, size(a))
    b[2, 1, 1] = 1
    @assert lp01_fidelity(a, b) == 0.0
    rejected_zero = try
        lp01_fidelity(a, zero(a))
        false
    catch err
        err isa ArgumentError
    end
    @assert rejected_zero
end
println("LP01 overlap definition checks passed.")

## 4. Check the starting point and the numerical gradient

Compare 10 mm and 1 mm propagation steps and compare slopes from ±200 W and
±100 W. Each evaluation recalculates both candidate-dependent branches.
The DNL difference tolerance is dimensionless, not dB. These checks concern
the starting point within the fixed three-mode, 2048-sample model.

In [ ]:
baseline = dnl_at(reference_power_W)
baseline_fine = dnl_at(reference_power_W; dz=validation_dz)
g_start = power_gradient(1.0)
g_start_half = power_gradient(1.0; h=fd_step/2)
gradient_relative_change = abs(g_start-g_start_half)/max(abs(g_start_half), 1e-8)
baseline_step_error = abs(baseline.D_NL-baseline_fine.D_NL)
@assert baseline_step_error < dnl_step_tolerance "Reduce search_dz before optimization."
@assert gradient_relative_change < 0.01 "Refine the finite-difference step before optimization."
brightness_floor = minimum_brightness_fraction * baseline_fine.photons
@printf("Starting gradient: %.6f DNL/u; half-step: %.6f DNL/u (%.3f%% change)\n",
    g_start, g_start_half, 100gradient_relative_change)
@printf("10 mm versus 1 mm starting objective DNL difference: %.8f\n", baseline_step_error)

## 5. Projected gradient descent with backtracking

Keep the original algorithm: propose \(u_{\rm new}=\operatorname{clip}(u-\eta g)\),
cap the power change, and halve it if the trial loses too many LP01 photons or
does not decrease DNL sufficiently. Save accepted steps only.

Stop on a small projected gradient, failed line search, or the iteration/evaluation
budget. A budget stop does not establish a minimum.

In [ ]:
optimization = optimize_power()
println("Stop: ", optimization.status)
@printf("Best accepted power: %.4f kW; objective: DNL=%.8f\n",
    optimization.best.power_W/1000, optimization.best.D_NL)

## 6. Validate the selected result

Recompute the selected power at 1 mm and 5 mm and test ±500 W neighbors at 5 mm.
Compare baseline and selected DNL at the same 1 mm resolution. Check improvement,
LP01 brightness, step-size agreement, tested neighbors, and decreasing history.

Passing supports an improvement in this fixed model, not a global optimum,
convergence in time samples/modes, an experimental prediction, or pulse cleanliness.

In [ ]:
best_power_W = optimization.best.power_W
best_fine = dnl_at(best_power_W; dz=validation_dz)
best_mid = dnl_at(best_power_W; dz=search_dz/2)
neighbor_delta_W = 500.0
left_neighbor = dnl_at(max(power_bounds_W[1], best_power_W-neighbor_delta_W); dz=search_dz/2)
right_neighbor = dnl_at(min(power_bounds_W[2], best_power_W+neighbor_delta_W); dz=search_dz/2)

improvement_DNL = baseline_fine.D_NL-best_fine.D_NL
best_step_error = abs(optimization.best.D_NL-best_fine.D_NL)
@assert improvement_DNL > 0 "No validated improvement over the starting point."
@assert best_fine.photons >= brightness_floor "Brightness constraint fails on the finer grid."
@assert best_step_error < dnl_step_tolerance "Refine search_dz and rerun optimization."
@assert best_mid.D_NL <= min(left_neighbor.D_NL, right_neighbor.D_NL)
@assert all(diff([r.D_NL for r in optimization.history]) .< 0)

summary = Dict(
    "status"=>optimization.status,
    "baseline_power_W"=>reference_power_W, "best_power_W"=>best_power_W,
    "baseline_D_NL"=>baseline_fine.D_NL,
    "best_D_NL"=>best_fine.D_NL,
    "baseline_F_NL"=>baseline_fine.F_NL, "best_F_NL"=>best_fine.F_NL,
    "improvement_DNL"=>improvement_DNL,
    "relative_DNL_reduction"=>1-best_fine.D_NL/baseline_fine.D_NL,
    "baseline_photons"=>baseline_fine.photons, "best_photons"=>best_fine.photons,
    "brightness_ratio"=>best_fine.photons/baseline_fine.photons,
    "best_lp01_energy_fraction"=>best_fine.lp01_energy_fraction,
    "best_mode2_energy_fraction"=>best_fine.mode2_energy_fraction,
    "best_mode3_energy_fraction"=>best_fine.mode3_energy_fraction,
    "baseline_F_input"=>baseline_fine.F_input, "best_F_input"=>best_fine.F_input,
    "baseline_step_error"=>baseline_step_error,
    "best_step_error"=>best_step_error,
    "gradient_step_relative_change"=>gradient_relative_change,
    "accepted_steps"=>length(optimization.history)-1,
    "optimization_seconds"=>optimization.seconds,
    "new_optimizer_objective_evaluations"=>optimization.new_objective_evaluations,
    "cp_relative_error"=>S_cp.relative_error,
    "validation_passed"=>true,
    "validation_scope"=>"fixed three-mode model; propagation-step and tested-neighbor checks",
)
open(joinpath(run_dir, "summary.toml"), "w") do io
    TOML.print(io, summary)
end
@printf("Validated: %.3f → %.3f kW; DNL %.8f → %.8f\n",
    reference_power_W/1000, best_power_W/1000,
    baseline_fine.D_NL, best_fine.D_NL)
@printf("DNL reduction: %.2f%%; LP01 brightness retained: %.2f%%\n",
    100summary["relative_DNL_reduction"], 100summary["brightness_ratio"])
@printf("Optimization: %.1f s, %d accepted steps, %d new objective evaluations\n",
    optimization.seconds, summary["accepted_steps"], optimization.new_objective_evaluations)

## 7. Look at the power dependence and optimization path

Sweep the entire allowed 50–150 kW range with 51 equally spaced powers (2 kW
spacing), alongside the accepted optimization path. The sweep is evaluated
after optimization and does not select the result. Crosses identify the 1 mm
validation values; the sweep and search values use 10 mm propagation steps.

In [ ]:
scan_lower, scan_upper = power_bounds_W
scan = [dnl_at(P) for P in range(scan_lower, scan_upper; length=51)]
save_rows("power_sweep.csv", scan)

fig, axes = subplots(1, 2; figsize=(9, 3.5), constrained_layout=true)
axes[1].plot([r.power_W/1000 for r in scan], [r.D_NL for r in scan],
    "o-"; color="#366b9e", label="Full power sweep")
axes[1].plot([r.power_W/1000 for r in optimization.history],
    [r.D_NL for r in optimization.history], "s--";
    color="#cf7c32", label="Accepted gradient steps", markersize=4)
axes[1].scatter([reference_power_W/1000, best_power_W/1000],
    [baseline_fine.D_NL, best_fine.D_NL];
    color="black", marker="x", s=60, label="1 mm validation", zorder=5)
axes[1].set_xlabel("Input peak power (kW)")
axes[1].set_ylabel("LP01 nonlinear distortion DNL")
axes[1].set_title("One input: peak power")
axes[1].legend(; fontsize=8)
axes[2].plot([r.iteration for r in optimization.history],
    [r.D_NL for r in optimization.history], "o-"; color="#366b9e")
axes[2].set_xlabel("Accepted iteration")
axes[2].set_ylabel("LP01 nonlinear distortion DNL")
axes[2].set_title("Gradient descent: lower is better")
for ax in axes
    ax.grid(; alpha=0.2)
    ax.spines["top"].set_visible(false)
    ax.spines["right"].set_visible(false)
end
savefig(joinpath(run_dir, "dnl_power_optimization.png"); dpi=180)
savefig(joinpath(run_dir, "dnl_power_optimization.pdf"))
display(fig)
close(fig)
@printf("Total objective evaluations (setup, search, validation, plot): %d\n", length(evaluations))
@printf("Total measured objective time: %.1f s\n", sum(r.seconds for r in evaluations))

## 8. Inspect the baseline and selected LP01 output

For each input power, show the time-domain pulse and the wavelength spectrum:

- Time (ps) versus LP01 power (kW).
- Wavelength (nm) versus power per frequency bin (W), matching
  step_index_squeezing.ipynb.

The spectrum plots frequency-bin powers at their associated wavelengths.
Its vertical axis is **not energy per nanometre**. Input, matched linear
output, and nonlinear output use the same 1 mm resolution. The curves use
absolute units. Display limits zoom to occupied support at a relative threshold
of 10⁻⁵; full fields remain in the NPZ files.

In [ ]:
times_ps = collect(time_axis(time_grid))
detuning_THz = collect(frequency_axis(time_grid)) ./ (2pi)
wavelengths_nm = collect(wavelength_axis(time_grid, properties.omega0)) .* 1e9
wavelength_indices = findall(isfinite.(wavelengths_nm) .& (wavelengths_nm .> 0))
wavelength_indices = wavelength_indices[sortperm(wavelengths_nm[wavelength_indices])]

plot_lp01_output("baseline", baseline_fine)
plot_lp01_output("selected", best_fine)
println("Saved time plots, wavelength spectra, and endpoint fields to ", run_dir)

## 9. Take a peek at the spatial modes, input and output

First, display the three fixed mode shapes in the same style as the original
notebook. Red and blue indicate opposite signs of the mode's field amplitude.
These shapes form the spatial basis used to propagate the pulse.
The dashed circle is the fiber core; both axes are positions in micrometres.

In [ ]:
x = properties.modes.x
y = properties.modes.y
modes = properties.modes.fields  # axes: y, x, mode
mode_names = ("LP01", "LP11 — cos orientation", "LP11 — sin orientation")

let fig = figure(; figsize=(18, 6), constrained_layout=true)
    for ii = 1:3
        ax = subplot(1, 3, ii)
        # Normalize amplitude only for displaying the fixed mode shapes.
        profile = real.(modes[:, :, ii])
        profile = profile ./ maximum(abs, profile)
        mesh = ax.pcolormesh(x.*1e6, y.*1e6, profile;
            cmap="seismic", vmin=-1, vmax=1, shading="auto", rasterized=true)
        ax.set_title("Mode $ii: "*mode_names[ii])
        ax.set_xlabel("x (μm)")
        ax.set_ylabel("y (μm)")
        ax.set_aspect("equal")
        theta = range(0, 2pi; length=181)
        ax.plot(settings["core_radius_m"].*1e6.*cos.(theta),
            settings["core_radius_m"].*1e6.*sin.(theta);
            color="#555555", linestyle="--", linewidth=0.8)
        fig.colorbar(mesh; ax, shrink=0.75, label="Relative field amplitude")
    end
    fig.suptitle("The three spatial modes — dashed line marks the fiber core")
    savefig(joinpath(run_dir, "spatial_modes.png"); dpi=160)
    savefig(joinpath(run_dir, "spatial_modes.pdf"))
    display(fig)
    close(fig)
end
npzwrite(joinpath(run_dir, "spatial_mode_fields.npz"),
    Dict("x_m"=>x, "y_m"=>y, "modes_y_x_m"=>modes))

Now combine the saved complex mode amplitudes with these spatial mode
shapes to display **input, linear-reference, and nonlinear-output intensity**.
Use all three propagated modes, even though the DNL objective scores only LP01.

Each panel shows its pulse at **its own peak-power time**, printed in the title.
**Every beam-intensity panel uses the same fixed 0–100 W/μm² color scale,**
shared by the baseline, selected result, and both launch-optimization notebooks.
Complex mode amplitudes are summed before squaring, preserving interference.
Integrating the spatial map is checked against the pulse power at that time.

The nearly pure LP01 launch should give similarly round beam shapes here.
Changes in duration, spectrum and phase can still occur; the temporal and
spectral plots above show complementary views. These previews use cached
endpoint fields and need no further propagation solves.

The shared bounds are set by beam_intensity_limits_W_per_um2 below. If a future result exceeds 100 W/μm², increase the limit in all three notebooks together; the check prevents silently clipping a bright peak.

In [ ]:
# Use the same absolute intensity scale in all three DNL notebooks.
# This covers the saved baseline, selected, and individual-mode intensity maps.
beam_intensity_limits_W_per_um2 = (0.0, 100.0)
beam_color_ticks = collect(range(beam_intensity_limits_W_per_um2...; length=6))

plot_beam_preview("baseline", baseline_fine)
plot_beam_preview("selected", best_fine)